# 01 — Data Collection
**Neural Voice Identity Control & Deepfake Audio Analysis — Disertație 2026**

Acest notebook:
1. Instalează dependențele (yt-dlp, demucs, ffmpeg)
2. Descarcă audio de pe YouTube pentru fiecare voce
3. Separă vocea de muzică/zgomot cu Demucs
4. Converteste la 40kHz mono WAV (format RVC)
5. Taie automat pauzele și produce clipuri de 5-10s gata pentru antrenare

---
**Rulează celulele în ordine de sus în jos.**

In [ ]:
# Mount Google Drive (checkpoint-urile și datele se salvează persistent)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Instalare dependențe
!pip install -q yt-dlp pydub librosa soundfile
!pip install -q demucs
!apt-get install -q -y ffmpeg
print('Dependencies installed.')

In [ ]:
# Clone repo (pentru a folosi scripturile)
import os

REPO_URL = 'https://github.com/vio688/disertatie-voice-deepfake.git'
REPO_DIR = '/content/disertatie'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## Configurare voci

**TODO:** Completează dicționarul de mai jos cu:
- Numele real al fiecărei persoane
- URL-urile YouTube (interviuri, podcast-uri, discursuri publice — **fără muzică în fundal**)

Sugestii surse de calitate bună:
- Interviuri la podcast-uri (audio curat, o singură persoană vorbind)
- Discursuri publice (fără aplauze intense)
- Conferințe de presă

Evită: videoclipuri cu muzică în fundal, mai mulți vorbitori simultan, calitate audio slabă.

In [ ]:
# ============================================================
# CONFIGURARE
# ============================================================
VOICES = {
    'voice_1': {
        'name': 'Călin Georgescu',
        'youtube_urls': [],  # lasă gol — uploadezi tu audio pe Drive
    },
    'voice_2': {
        'name': 'Nicușor Dan',
        'youtube_urls': [],
    },
    'voice_3': {
        'name': 'Diana Șoșoacă',
        'youtube_urls': [],
    },
}

# Paths pe Google Drive
DRIVE_BASE = '/content/drive/MyDrive/disertatie'
RAW_DIR = f'{DRIVE_BASE}/data/raw'
PROCESSED_DIR = f'{DRIVE_BASE}/data/processed'

import os
for voice_id in VOICES:
    os.makedirs(f'{RAW_DIR}/{voice_id}', exist_ok=True)
    os.makedirs(f'{PROCESSED_DIR}/{voice_id}/rvc_ready', exist_ok=True)
    os.makedirs(f'{PROCESSED_DIR}/{voice_id}/clips', exist_ok=True)
print('Directories ready.')

In [ ]:
import subprocess
from pathlib import Path

def download_voice(voice_id, config):
    """Download audio from YouTube using yt-dlp."""
    out_dir = f'{RAW_DIR}/{voice_id}'
    print(f"\n=== {config['name']} ({voice_id}) ===")
    
    for url in config['youtube_urls']:
        print(f'  Downloading: {url}')
        result = subprocess.run([
            'yt-dlp',
            '--extract-audio', '--audio-format', 'wav', '--audio-quality', '0',
            '--output', f'{out_dir}/%(title).50s.%(ext)s',
            '--no-playlist',
            url,
        ], capture_output=True, text=True)
        if result.returncode == 0:
            print(f'  ✓ Downloaded')
        else:
            print(f'  ✗ Error: {result.stderr[-200:]}')

# Download all voices
for voice_id, config in VOICES.items():
    if config['youtube_urls']:
        download_voice(voice_id, config)
    else:
        print(f'  Skipping {voice_id}: no URLs configured')

print('\nDownload complete.')

In [ ]:
import subprocess
from pathlib import Path

def separate_vocals(voice_id):
    """Extract vocals using Demucs htdemucs_ft model."""
    input_dir = Path(f'{RAW_DIR}/{voice_id}')
    output_dir = Path(f'{PROCESSED_DIR}/{voice_id}/demucs_out')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    wav_files = list(input_dir.glob('*.wav'))
    if not wav_files:
        print(f'  No .wav files in {input_dir}')
        return
    
    print(f'  Found {len(wav_files)} files to process')
    for wav in wav_files:
        print(f'  Separating: {wav.name}  (this may take a few minutes...)')
        subprocess.run([
            'python', '-m', 'demucs',
            '--two-stems=vocals',
            '--model', 'htdemucs_ft',
            '--out', str(output_dir),
            str(wav),
        ], check=True)
    print(f'  ✓ Demucs done for {voice_id}')

for voice_id in VOICES:
    print(f'\n=== Separating vocals: {voice_id} ===')
    separate_vocals(voice_id)

print('\nVocal separation complete.')

In [ ]:
import subprocess
from pathlib import Path

def convert_to_rvc_format(voice_id):
    """Convert separated vocals to 40kHz mono WAV (RVC v2 format)."""
    demucs_out = Path(f'{PROCESSED_DIR}/{voice_id}/demucs_out')
    rvc_ready = Path(f'{PROCESSED_DIR}/{voice_id}/rvc_ready')
    rvc_ready.mkdir(parents=True, exist_ok=True)
    
    # Demucs creates: demucs_out/{model}/{stem}/vocals.wav
    vocals_files = list(demucs_out.rglob('vocals.wav'))
    if not vocals_files:
        print(f'  No vocals.wav found under {demucs_out}')
        return
    
    for v in vocals_files:
        out_file = rvc_ready / f'{v.parent.name}.wav'
        subprocess.run([
            'ffmpeg', '-i', str(v),
            '-ar', '40000', '-ac', '1', '-acodec', 'pcm_s16le',
            str(out_file), '-y',
        ], check=True, capture_output=True)
    
    count = len(list(rvc_ready.glob('*.wav')))
    print(f'  ✓ {count} files → {rvc_ready}')

for voice_id in VOICES:
    print(f'\n=== Converting: {voice_id} ===')
    convert_to_rvc_format(voice_id)

print('\nConversion complete.')

## Preprocesare automată — tăiere pauze + split clipuri 5-10s

Această celulă:
- Detectează și elimină pauzele (silence) automat
- Normalizează volumul la -20 dBFS
- Produce clipuri de 5-10 secunde gata pentru RVC training


In [ ]:
from pathlib import Path
from pydub import AudioSegment
from pydub.silence import split_on_silence
import librosa

TARGET_DBFS = -20.0
MIN_CLIP_S = 4.0
MAX_CLIP_S = 10.0
MIN_SILENCE_MS = 400
SAMPLE_RATE = 40000

def preprocess_voice_clips(voice_id):
    input_dir = Path(f'{PROCESSED_DIR}/{voice_id}/rvc_ready')
    output_dir = Path(f'{PROCESSED_DIR}/{voice_id}/clips')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    wav_files = sorted(input_dir.glob('*.wav'))
    if not wav_files:
        print(f'  No .wav files in {input_dir}')
        return
    
    total_clips = 0
    for file_idx, wav_file in enumerate(wav_files):
        print(f'  [{file_idx+1}/{len(wav_files)}] {wav_file.name}')
        audio = AudioSegment.from_wav(str(wav_file))
        
        # Normalize
        audio = audio.apply_gain(TARGET_DBFS - audio.dBFS)
        
        # Split on silence
        chunks = split_on_silence(
            audio,
            min_silence_len=MIN_SILENCE_MS,
            silence_thresh=audio.dBFS - 16,
            keep_silence=150,
            seek_step=10,
        )
        
        clip_idx = 0
        buffer = AudioSegment.empty()
        
        for chunk in chunks:
            buffer += chunk
            dur_s = len(buffer) / 1000.0
            
            if dur_s >= MIN_CLIP_S:
                cut = min(int(MAX_CLIP_S * 1000), len(buffer))
                clip = buffer[:cut]
                fname = output_dir / f'{voice_id}_{file_idx:04d}_{clip_idx:04d}.wav'
                clip.export(str(fname), format='wav', parameters=['-ar', str(SAMPLE_RATE), '-ac', '1'])
                clip_idx += 1
                buffer = buffer[cut:]
        
        # Flush remainder
        if len(buffer) / 1000.0 >= MIN_CLIP_S:
            fname = output_dir / f'{voice_id}_{file_idx:04d}_{clip_idx:04d}.wav'
            buffer.export(str(fname), format='wav', parameters=['-ar', str(SAMPLE_RATE), '-ac', '1'])
            clip_idx += 1
        
        print(f'    → {clip_idx} clips')
        total_clips += clip_idx
    
    # Summary
    all_clips = list(output_dir.glob('*.wav'))
    total_min = sum(librosa.get_duration(path=str(f)) for f in all_clips) / 60.0
    print(f'\n  Total: {total_clips} clips, {total_min:.1f} minutes')
    if total_min < 10:
        print('  ⚠ WARNING: mai puțin de 10 minute. Calitatea fine-tuning-ului va fi afectată.')
        print('    Recomandat: 15-30 minute audio curat per voce.')

for voice_id in VOICES:
    print(f'\n=== Preprocessing clips: {VOICES[voice_id]["name"]} ({voice_id}) ===')
    preprocess_voice_clips(voice_id)

print('\n✓ Preprocessing complete! Continuă cu: 02_rvc_finetune.ipynb')

In [ ]:
# Verificare finală — listează clipurile produse
from pathlib import Path
import librosa

print('=== Rezumat date colectate ===\n')
for voice_id, config in VOICES.items():
    clips_dir = Path(f'{PROCESSED_DIR}/{voice_id}/clips')
    clips = list(clips_dir.glob('*.wav'))
    if clips:
        total_s = sum(librosa.get_duration(path=str(f)) for f in clips)
        print(f'{config["name"]} ({voice_id})')
        print(f'  Clipuri: {len(clips)}')
        print(f'  Durată totală: {total_s/60:.1f} min')
        print(f'  Durată medie clip: {total_s/len(clips):.1f}s')
    else:
        print(f'{voice_id}: NU EXISTĂ CLIPURI (verifică pașii anteriori)')
    print()